# Task 2 - detection

Needs Task 1 done (`configs/data.yaml` present). **Run All**, or run B0 first and
the training cell when you have time for it.

| Run | Model | What it is for |
| --- | --- | --- |
| B0 | COCO weights, no fine-tune | the floor. Finds only stop signs, so cone and barrier AP are 0 by construction |
| B1 | `yolo26n` fine-tuned, 640 | main deployable candidate |
| B1b | `yolo11n` fine-tuned, 640 | other family, for the latency pick in Task 4 |
| B2 | `yolo26s` fine-tuned, 960 | accuracy ceiling |

Every variant is scored by the same evaluator, including B0 - `model.val()` cannot
score an 80-class model against a 3-class set, and using two different metrics
would make the comparison meaningless.

In [ ]:
# ---- edit this ----
RUNS = ["B1"]        # add "B1b", "B2" when you want them
EPOCHS = 100         # patience=20 usually stops it earlier
WORKERS = 6          # drop to 2 if you hit WinError 1455 (paging file too small)
DEVICE = ""          # "" = auto, or "cpu", or "0"
# -------------------

In [ ]:
import os, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))

from common.config import load_yaml

data_yaml = REPO / "configs/data.yaml"
assert data_yaml.exists(), "run the Task 1 notebook first"
DEVICE_ARG = f"--device {DEVICE}" if DEVICE else ""
RUN_SPECS = load_yaml("runs.yaml")

print(data_yaml.read_text())
from IPython.display import Image, display

In [ ]:
# The floor. Everything below has to beat the stop sign number.
!python -u -m eval.det_eval --run B0 {DEVICE_ARG}

In [ ]:
# Train and score each run. exist_ok=True keeps the weights at runs/<name>/ so a
# re-run replaces them instead of creating runs/B12, runs/B13 and so on.
#
# python -u keeps the child process from block-buffering its output, so the
# epoch table appears as it happens. For the cleanest view, run the same command
# in a terminal and use the history cell below to follow along.
for run in RUNS:
    spec = {**RUN_SPECS["defaults"], **RUN_SPECS["runs"][run]}
    imgsz = spec["imgsz"]
    print(f"\n=== {run}: {spec.get('purpose', '')} ===")
    !python -u -m detection.train --run {run} --set epochs={EPOCHS} --set workers={WORKERS} --set exist_ok=True {DEVICE_ARG}
    !python -u -m eval.det_eval --weights runs/{run}/weights/best.pt --name {run} --imgsz {imgsz} {DEVICE_ARG} --cross-check

In [ ]:
# Per-epoch history. Ultralytics appends a row after every epoch, so this also
# works as a progress monitor: if you started training in a terminal instead,
# re-run this cell whenever you want to see where it is.
import pandas as pd

hist = pd.read_csv(f"runs/{RUNS[0]}/results.csv")
hist.columns = [c.strip() for c in hist.columns]
cols = [c for c in hist.columns if "mAP50" in c or "loss" in c]
print(f"{len(hist)} epochs done")
display(hist[["epoch", *cols]].tail(12))
display(Image(f"runs/{RUNS[0]}/results.png", width=1000))

In [ ]:
# Everything scored so far.
import pandas as pd

df = pd.read_csv("results/detection.csv")
display(df)

In [ ]:
# Per-class detail, including recall by ground-truth depth - this is where far
# cones either show up or do not, and it drives the input-size decision.
import json

for run in RUNS:
    path = REPO / f"results/det_eval_{run}.json"
    if not path.exists():
        continue
    detail = json.loads(path.read_text())
    print(f"\n{run}   mAP50 {detail['mAP50']:.3f}   mAP50-95 {detail['mAP50_95']:.3f}")
    for name, m in detail["per_class"].items():
        print(f"  {name:<10} AP50 {m['AP50']:.3f}  AP50-95 {m['AP50_95']:.3f}  gt {m['gt_instances']}")
        for rng, v in m["recall_by_range"].items():
            if v["recall"] is not None:
                print(f"{'':<14}{rng:<10} recall {v['recall']:.2f}  (n={v['gt']})")

In [ ]:
# What it actually draws on held-out frames.
from IPython.display import Image, display

from common.env import dataset_root_for_cli

run = RUNS[0]
VAL_IMAGES = dataset_root_for_cli() / "images" / "val"
!python -u -m detection.infer --weights runs/{run}/weights/best.pt --source "{VAL_IMAGES}" --limit 6 --dest results/predictions/{run} {DEVICE_ARG}

for p in sorted((REPO / f"results/predictions/{run}").glob("*.jpg"))[:6]:
    display(Image(str(p), width=900))

## Done when

- every fine-tuned run beats B0 on **all three** classes (B0 was stop sign AP50 0.745);
- `--cross-check` agrees with the custom evaluator within a point or two - small gaps are
  expected, ultralytics letterboxes and rectangles differently;
- stop sign AP has not collapsed. If it has, suspect the pseudo-labels from Task 1.

`runs/B1/weights/best.pt` is what Tasks 3 and 4 start from. Weights are git-ignored, so
copy that file by hand if the next task runs on the other machine.